# 시계열 예측 분석 - Chapter 3

## 📋 목차
1. [환경 설정](#환경-설정)
2. [데이터 로드 및 전처리](#데이터-로드-및-전처리)
3. [시계열 데이터 시각화](#시계열-데이터-시각화)
4. [추세 및 계절성 분석](#추세-및-계절성-분석)
5. [정상성 검정](#정상성-검정)
6. [이동평균 분석](#이동평균-분석)
7. [실습 예제](#실습-예제)

---

## 1. 환경 설정

필요한 라이브러리를 import합니다.

In [ ]:
# 기본 라이브러리
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# 시계열 분석 라이브러리
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.stattools import adfuller, kpss
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

# 시각화 설정
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (15, 6)
plt.rcParams['font.size'] = 12

# 한글 폰트 설정 (필요시)
# plt.rcParams['font.family'] = 'Malgun Gothic'  # Windows
# plt.rcParams['font.family'] = 'AppleGothic'    # macOS
plt.rcParams['axes.unicode_minus'] = False

print("✅ 라이브러리 import 완료")
print(f"pandas version: {pd.__version__}")
print(f"numpy version: {np.__version__}")

## 2. 데이터 로드 및 전처리

### 2.1 샘플 데이터 생성

In [ ]:
# 샘플 시계열 데이터 생성
np.random.seed(42)

# 날짜 범위 생성
dates = pd.date_range('2020-01-01', periods=365, freq='D')

# 추세 + 계절성 + 노이즈
trend = np.linspace(100, 150, 365)
seasonal = 10 * np.sin(np.linspace(0, 4*np.pi, 365))
noise = np.random.normal(0, 3, 365)
values = trend + seasonal + noise

# DataFrame 생성
df = pd.DataFrame({'value': values}, index=dates)

print("데이터 생성 완료!")
print(f"데이터 크기: {df.shape}")
print(f"기간: {df.index.min()} ~ {df.index.max()}")

### 2.2 데이터 기본 정보 확인

In [ ]:
# 기본 정보
print("=" * 60)
print("데이터 기본 정보")
print("=" * 60)
print(df.info())

print("\n" + "=" * 60)
print("데이터 통계 요약")
print("=" * 60)
print(df.describe())

print("\n" + "=" * 60)
print("결측치 확인")
print("=" * 60)
print(f"결측치 개수: {df.isnull().sum().sum()}")

In [ ]:
# 데이터 미리보기
print("처음 10개 데이터:")
display(df.head(10))

print("\n마지막 10개 데이터:")
display(df.tail(10))

## 3. 시계열 데이터 시각화

### 3.1 기본 시계열 플롯

In [ ]:
plt.figure(figsize=(15, 6))
plt.plot(df.index, df['value'], linewidth=2, color='steelblue')
plt.title('시계열 데이터 시각화', fontsize=18, fontweight='bold', pad=20)
plt.xlabel('날짜', fontsize=14)
plt.ylabel('값', fontsize=14)
plt.grid(True, alpha=0.3, linestyle='--')
plt.tight_layout()
plt.show()

### 3.2 다중 서브플롯 시각화

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(15, 12))

# 원본 데이터
axes[0].plot(df.index, df['value'], color='blue', linewidth=1.5)
axes[0].set_title('원본 시계열 데이터', fontsize=14, fontweight='bold')
axes[0].set_ylabel('값', fontsize=12)
axes[0].grid(True, alpha=0.3)

# 히스토그램
axes[1].hist(df['value'], bins=50, color='skyblue', edgecolor='black', alpha=0.7)
axes[1].set_title('데이터 분포', fontsize=14, fontweight='bold')
axes[1].set_xlabel('값', fontsize=12)
axes[1].set_ylabel('빈도', fontsize=12)
axes[1].grid(True, alpha=0.3, axis='y')

# Box Plot
axes[2].boxplot(df['value'], vert=False, patch_artist=True,
                boxprops=dict(facecolor='lightblue', alpha=0.7))
axes[2].set_title('Box Plot (이상치 확인)', fontsize=14, fontweight='bold')
axes[2].set_xlabel('값', fontsize=12)
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 4. 추세 및 계절성 분석

### 4.1 시계열 분해 (Seasonal Decomposition)

In [ ]:
# 시계열 분해 수행
decomposition = seasonal_decompose(df['value'], 
                                   model='additive',  # 'multiplicative'도 가능
                                   period=30)  # 주기 설정

print("✅ 시계열 분해 완료")

In [ ]:
# 시각화
fig, axes = plt.subplots(4, 1, figsize=(15, 14))

# 원본 데이터
decomposition.observed.plot(ax=axes[0], color='blue', linewidth=1.5)
axes[0].set_ylabel('원본', fontsize=12, fontweight='bold')
axes[0].set_title('시계열 분해 분석 (Additive Model)', 
                  fontsize=16, fontweight='bold', pad=15)
axes[0].grid(True, alpha=0.3)
axes[0].legend(['Observed'], loc='upper right')

# 추세 (Trend)
decomposition.trend.plot(ax=axes[1], color='red', linewidth=2)
axes[1].set_ylabel('추세', fontsize=12, fontweight='bold')
axes[1].grid(True, alpha=0.3)
axes[1].legend(['Trend'], loc='upper right')

# 계절성 (Seasonality)
decomposition.seasonal.plot(ax=axes[2], color='green', linewidth=1.5)
axes[2].set_ylabel('계절성', fontsize=12, fontweight='bold')
axes[2].grid(True, alpha=0.3)
axes[2].legend(['Seasonal'], loc='upper right')

# 잔차 (Residual)
decomposition.resid.plot(ax=axes[3], color='purple', linewidth=1)
axes[3].set_ylabel('잔차', fontsize=12, fontweight='bold')
axes[3].set_xlabel('날짜', fontsize=12)
axes[3].grid(True, alpha=0.3)
axes[3].axhline(y=0, color='black', linestyle='--', linewidth=1)
axes[3].legend(['Residual'], loc='upper right')

plt.tight_layout()
plt.show()

### 4.2 각 구성요소 통계

In [ ]:
print("=" * 60)
print("시계열 분해 구성요소 통계")
print("=" * 60)

print("\n[추세 통계]")
print(decomposition.trend.describe())

print("\n[계절성 통계]")
print(decomposition.seasonal.describe())

print("\n[잔차 통계]")
print(decomposition.resid.describe())

## 5. 정상성 검정

### 5.1 ADF Test 함수 정의

In [ ]:
def adf_test(series, name='시계열'):
    """
    ADF(Augmented Dickey-Fuller) 검정 수행
    
    귀무가설(H0): 시계열이 비정상성(단위근 존재)
    대립가설(H1): 시계열이 정상성
    """
    result = adfuller(series.dropna(), autolag='AIC')
    
    print(f'\n{"=" * 70}')
    print(f'ADF 검정 결과: {name}')
    print(f'{"=" * 70}')
    print(f'ADF Statistic      : {result[0]:.6f}')
    print(f'p-value            : {result[1]:.6f}')
    print(f'사용된 Lags        : {result[2]}')
    print(f'관측치 수          : {result[3]}')
    print(f'\nCritical Values:')
    for key, value in result[4].items():
        print(f'  {key:>4} : {value:.3f}')
    
    print(f'\n{"=" * 70}')
    print('해석:')
    if result[1] <= 0.05:
        print("✅ 귀무가설 기각 → 시계열이 정상성을 만족합니다.")
        print(f"   (p-value = {result[1]:.6f} ≤ 0.05)")
        print("   → 예측 모델 적용 가능")
    else:
        print("❌ 귀무가설 채택 → 시계열이 비정상성입니다.")
        print(f"   (p-value = {result[1]:.6f} > 0.05)")
        print("   → 차분 또는 변환 필요")
    print(f'{"=" * 70}\n')
    
    return result

print("✅ ADF Test 함수 정의 완료")

### 5.2 원본 데이터 정상성 검정

In [ ]:
print("🔍 원본 시계열 정상성 검정")
adf_result_original = adf_test(df['value'], '원본 시계열')

### 5.3 차분 후 정상성 검정

In [ ]:
# 1차 차분
df['diff_1'] = df['value'].diff()

print("🔍 1차 차분 시계열 정상성 검정")
adf_result_diff1 = adf_test(df['diff_1'], '1차 차분 시계열')

In [ ]:
# 2차 차분 (필요시)
df['diff_2'] = df['diff_1'].diff()

print("🔍 2차 차분 시계열 정상성 검정")
adf_result_diff2 = adf_test(df['diff_2'], '2차 차분 시계열')

### 5.4 차분 결과 시각화

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(15, 12))

# 원본
axes[0].plot(df.index, df['value'], color='blue', linewidth=1.5)
axes[0].set_title('원본 시계열', fontsize=14, fontweight='bold')
axes[0].set_ylabel('값', fontsize=12)
axes[0].grid(True, alpha=0.3)

# 1차 차분
axes[1].plot(df.index, df['diff_1'], color='green', linewidth=1.5)
axes[1].axhline(y=0, color='red', linestyle='--', linewidth=1)
axes[1].set_title('1차 차분', fontsize=14, fontweight='bold')
axes[1].set_ylabel('차분값', fontsize=12)
axes[1].grid(True, alpha=0.3)

# 2차 차분
axes[2].plot(df.index, df['diff_2'], color='purple', linewidth=1.5)
axes[2].axhline(y=0, color='red', linestyle='--', linewidth=1)
axes[2].set_title('2차 차분', fontsize=14, fontweight='bold')
axes[2].set_ylabel('차분값', fontsize=12)
axes[2].set_xlabel('날짜', fontsize=12)
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 6. 이동평균 분석

### 6.1 이동평균 계산

In [ ]:
# 다양한 윈도우 크기의 이동평균 계산
df['MA_7'] = df['value'].rolling(window=7).mean()    # 7일 이동평균
df['MA_30'] = df['value'].rolling(window=30).mean()  # 30일 이동평균
df['MA_90'] = df['value'].rolling(window=90).mean()  # 90일 이동평균

print("✅ 이동평균 계산 완료")
print(df[['value', 'MA_7', 'MA_30', 'MA_90']].head(100))

### 6.2 이동평균 시각화

In [ ]:
plt.figure(figsize=(15, 7))
plt.plot(df.index, df['value'], label='원본 데이터', linewidth=1, alpha=0.5, color='gray')
plt.plot(df.index, df['MA_7'], label='7일 이동평균', linewidth=2, color='blue')
plt.plot(df.index, df['MA_30'], label='30일 이동평균', linewidth=2, color='orange')
plt.plot(df.index, df['MA_90'], label='90일 이동평균', linewidth=2, color='red')
plt.title('이동평균을 통한 추세 분석', fontsize=16, fontweight='bold', pad=15)
plt.xlabel('날짜', fontsize=12)
plt.ylabel('값', fontsize=12)
plt.legend(fontsize=11, loc='best')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 7. 자기상관 분석 (ACF/PACF)

### 7.1 ACF (Autocorrelation Function)

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(15, 10))

# ACF
plot_acf(df['value'].dropna(), lags=50, ax=axes[0])
axes[0].set_title('자기상관함수 (ACF)', fontsize=14, fontweight='bold')
axes[0].grid(True, alpha=0.3)

# PACF
plot_pacf(df['value'].dropna(), lags=50, ax=axes[1])
axes[1].set_title('부분자기상관함수 (PACF)', fontsize=14, fontweight='bold')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 8. 실습 예제: 월별 데이터 분석

### 8.1 월별 데이터 생성

In [ ]:
# 월별 데이터 생성
np.random.seed(123)
dates_monthly = pd.date_range('2018-01-01', periods=60, freq='M')
trend_monthly = np.linspace(100, 200, 60)
seasonal_monthly = 20 * np.sin(np.linspace(0, 10*np.pi, 60))
noise_monthly = np.random.normal(0, 5, 60)
values_monthly = trend_monthly + seasonal_monthly + noise_monthly

df_monthly = pd.DataFrame({'sales': values_monthly}, index=dates_monthly)

print("✅ 월별 데이터 생성 완료")
print(df_monthly.head(12))

### 8.2 월별 데이터 시계열 분해

In [ ]:
# 시계열 분해
decomposition_monthly = seasonal_decompose(df_monthly['sales'], 
                                           model='additive', 
                                           period=12)

# 시각화
fig = plt.figure(figsize=(16, 12))
gs = fig.add_gridspec(4, 2, hspace=0.3, wspace=0.3)

# 원본
ax1 = fig.add_subplot(gs[0, :])
decomposition_monthly.observed.plot(ax=ax1, color='blue', linewidth=2)
ax1.set_title('원본 데이터 (월별 매출)', fontsize=16, fontweight='bold')
ax1.set_ylabel('Sales', fontsize=12)
ax1.grid(True, alpha=0.3)

# 추세
ax2 = fig.add_subplot(gs[1, :])
decomposition_monthly.trend.plot(ax=ax2, color='red', linewidth=2)
ax2.set_title('추세 (Trend)', fontsize=16, fontweight='bold')
ax2.set_ylabel('Trend', fontsize=12)
ax2.grid(True, alpha=0.3)

# 계절성
ax3 = fig.add_subplot(gs[2, :])
decomposition_monthly.seasonal.plot(ax=ax3, color='green', linewidth=2)
ax3.set_title('계절성 (Seasonality)', fontsize=16, fontweight='bold')
ax3.set_ylabel('Seasonal', fontsize=12)
ax3.grid(True, alpha=0.3)

# 잔차
ax4 = fig.add_subplot(gs[3, 0])
decomposition_monthly.resid.plot(ax=ax4, color='purple', linewidth=1)
ax4.axhline(y=0, color='black', linestyle='--')
ax4.set_title('잔차 (Residual)', fontsize=14, fontweight='bold')
ax4.set_ylabel('Residual', fontsize=12)
ax4.grid(True, alpha=0.3)

# 잔차 히스토그램
ax5 = fig.add_subplot(gs[3, 1])
decomposition_monthly.resid.dropna().hist(ax=ax5, bins=20, color='purple', 
                                          alpha=0.7, edgecolor='black')
ax5.set_title('잔차 분포', fontsize=14, fontweight='bold')
ax5.set_xlabel('Residual', fontsize=12)
ax5.set_ylabel('Frequency', fontsize=12)
ax5.grid(True, alpha=0.3, axis='y')

plt.show()

### 8.3 월별 계절성 패턴 분석

In [ ]:
# 계절성 패턴 추출
seasonal_pattern = decomposition_monthly.seasonal[:12]

print("\n" + "=" * 60)
print("월별 계절성 패턴")
print("=" * 60)
for i, value in enumerate(seasonal_pattern, 1):
    print(f"{i:2d}월: {value:+7.2f}")

# 시각화
plt.figure(figsize=(12, 6))
plt.bar(range(1, 13), seasonal_pattern, color='skyblue', edgecolor='black', alpha=0.7)
plt.axhline(y=0, color='red', linestyle='--', linewidth=1)
plt.title('월별 계절성 패턴', fontsize=16, fontweight='bold')
plt.xlabel('월', fontsize=12)
plt.ylabel('계절성 효과', fontsize=12)
plt.xticks(range(1, 13))
plt.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

## 9. 요약 및 결론

### 9.1 학습 내용 정리

In [ ]:
print("=" * 70)
print("Chapter 3 학습 내용 요약")
print("=" * 70)
print("\n✅ 완료한 내용:")
print("  1. 시계열 데이터 로드 및 전처리")
print("  2. 시계열 데이터 시각화")
print("  3. 시계열 분해 (추세, 계절성, 잔차)")
print("  4. 정상성 검정 (ADF Test)")
print("  5. 차분을 통한 정상성 확보")
print("  6. 이동평균 분석")
print("  7. 자기상관 분석 (ACF/PACF)")
print("\n📚 핵심 개념:")
print("  - 정상성 (Stationarity)")
print("  - 추세 (Trend)")
print("  - 계절성 (Seasonality)")
print("  - 차분 (Differencing)")
print("  - 이동평균 (Moving Average)")


### 9.2 데이터 저장 (선택사항)

In [ ]:
# 분석 결과 저장
# df.to_csv('timeseries_analysis_result.csv')
# print("✅ 분석 결과 저장 완료: timeseries_analysis_result.csv")

---

## 📚 참고 자료

- [Statsmodels Documentation](https://www.statsmodels.org/)
- [Pandas Time Series](https://pandas.pydata.org/docs/user_guide/timeseries.html)

---

**작성자**: Kim kwangseop  
**작성일**: 2026-05-01  
**GitHub**: [@yellowpart](https://github.com/yellowpart)